# Homework: Agentic RAG

The homework is available
[here](https://github.com/DataTalksClub/llm-zoomcamp/blob/main/cohorts/2026/01-agentic-rag/homework.md).

## Preparations

### Prepare Workspace

In [56]:
# dependencies
from gitsource import GithubRepositoryDataReader, chunk_documents
from minsearch import Index
from rag_helper import RAGBaseHomework
from anthropic import Anthropic
from toyaikit.llm import AnthropicClient
from toyaikit.tools import Tools
from toyaikit.chat.runners import AnthropicMessagesRunner, DisplayingRunnerCallback
from toyaikit.chat import IPythonChatInterface

### Import RAG Knowledge Base

We'll use the course's lessons.
The course is organized as a GitHub repo.
Lessons are Markdown files.

We use a particular commit (8c1834d) so everyone gets the same results for the
homework.

In [2]:
# initialize github repo reader
# let it read all lesson files: md files in lessons directories
reader = GithubRepositoryDataReader(
    repo_owner="DataTalksClub",
    repo_name="llm-zoomcamp",
    commit_id="8c1834d",
    allowed_extensions={"md"},
    filename_filter=lambda path: "/lessons/" in path,
)

In [3]:
# get the specified lesson files
files = reader.read()

In [4]:
# get a list of all files
# each files becomes a dictionary with filename and content

# initialize list for containing all files
documents = []

# call parse method to get the dictionary for each file
for file in files:
    doc = file.parse()
    documents.append(doc)

In [5]:
# inspect the loaded documents
print(f"{len(documents)} documents loaded.\n")

# print first 3 documents just to get an idea of what this looks like
# I mean I completed the lessons of the first chapter already
# but I want to see the data structure here in python
for document in documents[1:4]:
    print(document)

72 documents loaded.

{'content': '# Environment\n\nVideo: [Watch this lesson](https://www.youtube.com/watch?v=3U4gBrmkZyM&list=PL3MmuxUbc_hLZFNgSad56pDBKK8KO0XIv)\n\nFor this module, all you need is Python with Jupyter.\n\n## Prerequisites\n\nYou need the following:\n\n- Python (3.14 or later)\n- An [OpenAI account](https://openai.com/) (or an OpenAI-compatible\n  provider like Groq, Gemini, or Ollama)\n- Basic familiarity with Python and the command line\n\n## Creating the project\n\nWe\'ll start from scratch - no cloning needed. You\'ll create the\nproject yourself, step by step.\n\nFirst, install uv. It\'s a Python package manager, and I switched all my\nprojects to it because it\'s fast and convenient. Once I started using\nit, I never wanted to go back.\n\nOn Mac or Linux:\n\n```bash\ncurl -LsSf https://astral.sh/uv/install.sh | sh\n```\n\nOn Windows:\n\n```powershell\npowershell -ExecutionPolicy ByPass -c "irm https://astral.sh/uv/install.ps1 | iex"\n```\n\n(You can also use `pi

The first question is about how many documents are loaded like this.
The answer is 72.

### Indexing and Searching

In [6]:
# index the documents with minsearch
index = Index(
    # tokenize and search these as text
    text_fields=["content"],
    # use for literal filtering
    keyword_fields=["filename"],
)

# fit the index
index.fit(documents)

The next question is about which file shows up as first search result given a
particular query.

In [7]:
# perform the specified search
search_results = index.search(
    query="How does the agentic loop keep calling the model until it stops?"
)

search_results

[{'content': '# The Agentic Loop\n\nVideo: [Watch this lesson](https://www.youtube.com/watch?v=ePlQUcTPPjw&list=PL3MmuxUbc_hLZFNgSad56pDBKK8KO0XIv)\n\nIn the previous lesson, we did function calling by hand. We sent a\nmessage and got back a function call. We ran it, sent the result back,\nand got the answer.\n\nThat works for one function call. It breaks down when the model wants\nto search several times, or when the first search misses the answer.\nWe don\'t know in advance how many calls the model will want. So we\nneed a loop that keeps calling the model and running tools until it\'s\ndone. An agent is exactly that.\n\n## Anatomy of an agent\n\nWith the LLM in the driver\'s seat, we have an agent. It\'s an AI\nassistant whose goal is to help the user.\n\nAn agent has three parts:\n\n- Instructions, the role and behavior we want. We pass this as the\n  `developer` message. The better the instructions, the better the\n  agent helps.\n- Tools, the functions the agent can call to carry

In [8]:
# print the filename of the first result
print(f"The filename of the first result is: '{search_results[0]["filename"]}'")

The filename of the first result is: '01-agentic-rag/lessons/14-agentic-loop.md'


## RAG

Perform RAG on the lessons texts.

In [9]:
# initialize anthropic client for model responses
anthropic_client = Anthropic()

In [10]:
# initialize RAG assistant
assistant = RAGBaseHomework(
    index=index,
    llm_client=anthropic_client
)

In [11]:
# perform RAG
answer, input_tokens = assistant.rag(
    query="How does the agentic loop keep calling the model until it stops?"
)

In [12]:
# print answer
print(answer)

# How the Agentic Loop Keeps Calling the Model Until It Stops

The agentic loop uses a **`while True` loop with a simple exit condition** to keep calling the model until it stops needing tools.

## The Core Mechanism

Here's how it works:

```python
while True:
    print(f"iteration #{it}...")
    has_function_calls = False

    response = openai_client.responses.create(
        model="gpt-5.4-mini",
        input=messages,
        tools=[search_tool],
    )

    messages.extend(response.output)

    for item in response.output:
        if item.type == "function_call":
            # Execute the tool
            call_output = make_call(item)
            messages.append(call_output)
            has_function_calls = True
        elif item.type == "message":
            # Store the assistant's message
            last_answer = item.content[0].text

    it = it + 1
    if has_function_calls == False:
        break
```

## The Exit Condition

**The loop stops when `has_function_calls` is `Fa

The question is about the the number of input tokens sent to the
model in this request.
Print the result.

In [13]:
# print usage
print(f"{input_tokens} input tokens were sent.")

8088 input tokens were sent.


The homework's multiple choice doesn't contain 8088.
However, they use OpenAI, I use Anthropic.
Both of these use different tokenization algorithms, so the same
string will produce a different token count.
I will choose 7000, which is the closest available choice.
A ~15% difference should be within an expectable range.

## Chunking

In [15]:
# chunk documents
chunks = chunk_documents(documents, size=2000, step=1000)

In [20]:
# check how many chunks were produced for the next question
print(f"{len(chunks)} were made.")

295 were made.


In [ ]:
# index the chunked documents with minsearch
index = Index(
    # tokenize and search these as text
    text_fields=["content"],
    # use for literal filtering
    keyword_fields=["filename"],
)

# fit the index
index.fit(chunks)

In [22]:
# initialize RAG assistant pointing at indexed chunks
assistant = RAGBaseHomework(
    index=index,
    llm_client=anthropic_client
)

# perform RAG again
answer, input_tokens = assistant.rag(
    query="How does the agentic loop keep calling the model until it stops?"
)

# print usage to compare if chunking reduced number of input tokens sent
print(f"{input_tokens} input tokens were sent.")

2542 input tokens were sent.


Answering to the homework question: This is about 3x fewer.

## Making it Agentic with Toy AI Kit

In [57]:
# build a search function to be used as a tool by the agent
def search(query: str) -> dict[str, str]:
    """
    Search the FAQ database for entries matching the given query.
    """
    return index.search(query)

In [58]:
# define system prompt
INSTRUCTIONS = """
You're a course teaching assistant.
Answer the student's question using the search tool.
Make multiple searches with different keywords before answering.
"""

In [59]:
# register search as a tool
agent_tools = Tools()
agent_tools.add_tool(search)

In [60]:
# initialize chat interface for Jupyter notebooks
chat_interface = IPythonChatInterface()
callback = DisplayingRunnerCallback(chat_interface)

In [61]:
# initialize a tiny program (runner) for handling all the requests
runner = AnthropicMessagesRunner(
    tools=agent_tools,
    developer_prompt=INSTRUCTIONS,
    chat_interface=chat_interface,
    llm_client=AnthropicClient(model="claude-haiku-4-5"),
)

In [63]:
# ask the question from the exercise to the assistant
result = runner.loop(
    prompt="How does the agentic loop work, and how is it different from plain RAG?",
    callback=callback,
)

-> Response received


-> Response received


The homework asks how many times the agent called the search function.
In our case, it's three.